# 💳 Credit Default Prediction - Training Notebook
===================================================
Notebook ini digunakan untuk membangun, melatih, mengevaluasi, dan menyimpan model Deep Learning (TensorFlow) guna memprediksi kemungkinan nasabah gagal bayar kartu kredit (*credit card default*).

### 🛠️ Fitur & Teknik yang Diterapkan:
1. **TensorFlow Functional API**: Arsitektur jaringan saraf tiruan terstruktur.
2. **Custom Layer (`CustomDenseBlock`)**: Dense ➔ Batch Normalization ➔ ReLU ➔ Dropout dengan L2 regularization.
3. **Standard BinaryCrossentropy + Class Weighting**: Menangani ketidakseimbangan kelas menggunakan `class_weight` pada `model.fit()`.
4. **Custom Callback (`MetricsCallback`)**: Memantau F1-Score, Akurasi, dan MAE di setiap akhir epoch.
5. **31 Features (14 Base + 17 Engineered)**: Fitur demografis (SEX, EDUCATION, MARRIAGE, AGE) + finansial + rekayasa fitur.
6. **Fine-Tuning dengan GradientTape**: Optimasi lanjutan dengan *learning rate* kecil dan *gradient norm clipping*.

## Import Library

In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
import joblib
import os
import datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    classification_report,
    f1_score
)

print(f"TensorFlow Version: {tf.__version__}")

TensorFlow Version: 2.21.0


## 📊 STEP 1: Data Loading & Preprocessing
Membaca data dari `dataset_credit_final.csv` yang mencakup **fitur demografis** (SEX, EDUCATION, MARRIAGE, AGE) dan data finansial 3 bulan terakhir. Dilanjutkan dengan rekayasa fitur (*feature engineering*).

In [5]:
print("Loading dataset...")
df = pd.read_csv('./dataset_credit_final.csv')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Default rate: {df['default'].mean():.4f}")

# Fitur dasar: demografis + finansial 3 bulan terakhir
base_features = [
    'LIMIT_BAL',
    'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
    'PAY_1', 'PAY_2', 'PAY_3',
    'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3',
    'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3',
]

# --- Feature Engineering ---
df['total_bill'] = df['BILL_AMT1'] + df['BILL_AMT2'] + df['BILL_AMT3']
df['total_payment'] = df['PAY_AMT1'] + df['PAY_AMT2'] + df['PAY_AMT3']

df['debt_ratio'] = np.where(df['LIMIT_BAL'] == 0, 0, df['total_bill'] / df['LIMIT_BAL'])
df['avg_delay'] = df[['PAY_1', 'PAY_2', 'PAY_3']].mean(axis=1)
df['payment_ratio'] = np.where(df['total_bill'] == 0, 0, df['total_payment'] / df['total_bill'])
df['payment_to_limit_ratio'] = np.where(df['LIMIT_BAL'] == 0, 0, df['total_payment'] / df['LIMIT_BAL'])

df['max_delay'] = df[['PAY_1', 'PAY_2', 'PAY_3']].max(axis=1)
df['has_delay'] = (df[['PAY_1', 'PAY_2', 'PAY_3']].max(axis=1) > 0).astype(int)

df['utilization_1'] = np.where(df['LIMIT_BAL'] == 0, 0, df['BILL_AMT1'] / df['LIMIT_BAL'])
df['utilization_2'] = np.where(df['LIMIT_BAL'] == 0, 0, df['BILL_AMT2'] / df['LIMIT_BAL'])
df['utilization_3'] = np.where(df['LIMIT_BAL'] == 0, 0, df['BILL_AMT3'] / df['LIMIT_BAL'])

df['pay_bill_ratio_1'] = np.where(df['BILL_AMT1'] == 0, 0, df['PAY_AMT1'] / (np.abs(df['BILL_AMT1']) + 1))
df['pay_bill_ratio_2'] = np.where(df['BILL_AMT2'] == 0, 0, df['PAY_AMT2'] / (np.abs(df['BILL_AMT2']) + 1))
df['pay_bill_ratio_3'] = np.where(df['BILL_AMT3'] == 0, 0, df['PAY_AMT3'] / (np.abs(df['BILL_AMT3']) + 1))

df['bill_growth'] = np.where(np.abs(df['BILL_AMT3']) < 1, 0, (df['BILL_AMT1'] - df['BILL_AMT3']) / (np.abs(df['BILL_AMT3']) + 1))
df['payment_growth'] = np.where(np.abs(df['PAY_AMT3']) < 1, 0, (df['PAY_AMT1'] - df['PAY_AMT3']) / (np.abs(df['PAY_AMT3']) + 1))
df['delay_trend'] = df['PAY_1'] - df['PAY_3']

engineered_features = [
    'total_bill', 'total_payment', 'debt_ratio', 'avg_delay',
    'payment_ratio', 'payment_to_limit_ratio', 'max_delay', 'has_delay',
    'utilization_1', 'utilization_2', 'utilization_3',
    'pay_bill_ratio_1', 'pay_bill_ratio_2', 'pay_bill_ratio_3',
    'bill_growth', 'payment_growth', 'delay_trend'
]

all_features = base_features + engineered_features
print(f"\nBase features: {len(base_features)} (termasuk SEX, EDUCATION, MARRIAGE, AGE)")
print(f"Engineered features: {len(engineered_features)}")
print(f"Total features: {len(all_features)}")
print(f"Features: {all_features}")

X = df[all_features].copy()
y = df['default'].copy()

# Handle infinity dan NaN
X.replace([np.inf, -np.inf], 0, inplace=True)
X.fillna(0, inplace=True)

Loading dataset...
Dataset shape: (30000, 20)
Columns: ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_1', 'PAY_2', 'PAY_3', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'default', 'total_bill', 'total_payment', 'debt_ratio', 'avg_delay', 'payment_ratio']
Default rate: 0.2212

Base features: 14 (termasuk SEX, EDUCATION, MARRIAGE, AGE)
Engineered features: 17
Total features: 31
Features: ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_1', 'PAY_2', 'PAY_3', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'total_bill', 'total_payment', 'debt_ratio', 'avg_delay', 'payment_ratio', 'payment_to_limit_ratio', 'max_delay', 'has_delay', 'utilization_1', 'utilization_2', 'utilization_3', 'pay_bill_ratio_1', 'pay_bill_ratio_2', 'pay_bill_ratio_3', 'bill_growth', 'payment_growth', 'delay_trend']


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,BILL_AMT1,BILL_AMT2,...,has_delay,utilization_1,utilization_2,utilization_3,pay_bill_ratio_1,pay_bill_ratio_2,pay_bill_ratio_3,bill_growth,payment_growth,delay_trend
0,20000,2,2,1,24,2,2,-1,3913.0,3102.0,...,1,0.195650,0.155100,0.034450,0.000000,0.222043,0.000000,4.672464,0.000000,3
1,120000,2,2,2,26,-1,2,0,2682.0,1725.0,...,1,0.022350,0.014375,0.022350,0.000000,0.579374,0.372717,0.000000,-0.999001,-1
2,90000,2,2,2,34,0,0,0,29239.0,14027.0,...,0,0.324878,0.155856,0.150656,0.051915,0.106929,0.073746,1.156342,0.517483,0
3,50000,2,2,1,37,0,0,0,46990.0,48233.0,...,0,0.939800,0.964660,0.985820,0.042561,0.041858,0.024345,-0.046681,0.666112,0
4,50000,1,2,1,57,-1,0,-1,8617.0,5670.0,...,0,0.172340,0.113400,0.716700,0.232072,6.468171,0.279049,-0.759516,-0.799920,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,220000,1,3,1,39,0,0,0,188948.0,192815.0,...,0,0.858855,0.876432,0.947114,0.044986,0.103726,0.024011,-0.093187,0.698841,0
29996,150000,1,3,2,43,-1,-1,-1,1683.0,1828.0,...,0,0.011220,0.012187,0.023347,1.090855,1.927829,2.568655,-0.519269,-0.795755,0
29997,30000,1,2,2,37,3,3,2,3565.0,3356.0,...,1,0.118833,0.111867,0.091933,0.000000,0.000000,7.973904,0.292497,-0.999955,1
29998,80000,1,3,1,41,1,-1,0,-81.0,78379.0,...,1,-0.001012,0.979738,0.953800,811.246098,0.043493,0.015438,-1.001048,55.423393,1


### ⚖️ Train-Test Split dan Scaling
Data dibagi menjadi 80% train dan 20% test, kemudian di-scale menggunakan `StandardScaler`. Ketidakseimbangan kelas ditangani melalui parameter `class_weight` pada `model.fit()`.

In [6]:
# Train/test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train samples: {X_train.shape[0]} | Test samples: {X_test.shape[0]}")
print(f"Train default rate: {y_train.mean():.4f}")

# Scaling (fit hanya pada data Train)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Simpan Scaler dan Nama Fitur untuk kebutuhan API
joblib.dump(scaler, 'scaler.joblib')
joblib.dump(all_features, 'feature_names.joblib')
print("✓ Scaler dan feature names berhasil disimpan")

# Konversi ke Tensorflow Tensor (tanpa SMOTE, gunakan distribusi asli)
X_train_tensor = tf.convert_to_tensor(X_train_scaled, dtype=tf.float32)
X_test_tensor = tf.convert_to_tensor(X_test_scaled, dtype=tf.float32)
y_train_tensor = tf.convert_to_tensor(y_train.values, dtype=tf.float32)
y_test_tensor = tf.convert_to_tensor(y_test.values, dtype=tf.float32)

print(f"\nDistribusi kelas training (asli): {np.bincount(y_train)}")


Train samples: 24000 | Test samples: 6000
Train default rate: 0.2212
✓ Scaler dan feature names berhasil disimpan

Distribusi kelas training (asli): [18691  5309]


## 🧱 STEP 2: Komponen Custom TensorFlow
Mendefinisikan komponen custom: `CustomDenseBlock` (layer) dan `MetricsCallback` (callback pemantau training).

In [7]:
# --- Custom Layer ---
class CustomDenseBlock(tf.keras.layers.Layer):
    def __init__(self, units, dropout_rate=0.3, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.dropout_rate = dropout_rate

    def build(self, input_shape):
        self.dense = tf.keras.layers.Dense(
            self.units, kernel_initializer='he_normal',
            kernel_regularizer=tf.keras.regularizers.l2(1e-4)
        )
        self.batch_norm = tf.keras.layers.BatchNormalization()
        self.activation = tf.keras.layers.ReLU()
        self.dropout = tf.keras.layers.Dropout(self.dropout_rate)
        super().build(input_shape)

    def call(self, inputs, training=False):
        x = self.dense(inputs)
        x = self.batch_norm(x, training=training)
        x = self.activation(x)
        x = self.dropout(x, training=training)
        return x

    def get_config(self):
        config = super().get_config()
        config.update({'units': self.units, 'dropout_rate': self.dropout_rate})
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)


# --- Custom Callback ---
class MetricsCallback(tf.keras.callbacks.Callback):
    def __init__(self, X_val, y_val, **kwargs):
        super().__init__(**kwargs)
        self.X_val = X_val
        self.y_val = y_val
        self.best_accuracy = 0.0
        self.history = {'accuracy': [], 'mae': [], 'f1_score': []}

    def on_epoch_end(self, epoch, logs=None):
        val_predictions = self.model(self.X_val, training=False)
        val_pred_proba = tf.reshape(val_predictions, [-1]).numpy()
        val_pred_binary = (val_pred_proba > 0.5).astype(int)
        y_val_np = self.y_val.numpy().flatten() if hasattr(self.y_val, 'numpy') else np.array(self.y_val).flatten()
        
        accuracy = accuracy_score(y_val_np, val_pred_binary)
        mae = mean_absolute_error(y_val_np, val_pred_proba)
        f1 = f1_score(y_val_np, val_pred_binary, zero_division=0)

        self.history['accuracy'].append(accuracy)
        self.history['mae'].append(mae)
        self.history['f1_score'].append(f1)

        if accuracy > self.best_accuracy:
            self.best_accuracy = accuracy

        print(f"  >> Val Acc: {accuracy:.4f} | MAE: {mae:.4f} | F1: {f1:.4f} | Best: {self.best_accuracy:.4f}")


## 🏗️ STEP 3: Model Architecture (Functional API)
Membangun struktur jaringan neural network menggunakan TensorFlow Functional API dengan 4 Dense Block kustom. Input shape otomatis menyesuaikan jumlah fitur (31).

In [8]:
n_features = X_train_scaled.shape[1]

inputs = tf.keras.Input(shape=(n_features,), name='credit_input')
x = CustomDenseBlock(256, dropout_rate=0.3, name='block_1')(inputs)
x = CustomDenseBlock(128, dropout_rate=0.3, name='block_2')(x)
x = CustomDenseBlock(64, dropout_rate=0.25, name='block_3')(x)
x = CustomDenseBlock(32, dropout_rate=0.2, name='block_4')(x)

outputs = tf.keras.layers.Dense(1, activation='sigmoid', name='output')(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs, name='credit_default_model')
model.summary()

Model: "credit_default_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ credit_input (InputLayer)       │ (None, 31)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block_1 (CustomDenseBlock)      │ (None, 256)            │         9,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block_2 (CustomDenseBlock)      │ (None, 128)            │        33,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block_3 (CustomDenseBlock)      │ (None, 64)             │         8,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block_4 (CustomDenseBlock)      │ (None, 32)             │         2,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 53,377 (208.50 KB)

 Trainable params: 52,417 (204.75 KB)

 Non-trainable params: 960 (3.75 KB)

## 🏋️ STEP 4: Model Training
Melatih model menggunakan `model.fit()` dengan optimasi Adam, `BinaryCrossentropy`, `class_weight`, dan callback pemantau.

In [10]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

log_dir = os.path.join('logs', datetime.datetime.now().strftime('%Y%m%d-%H%M%S'))

callbacks = [
    MetricsCallback(X_val=X_test_tensor, y_val=y_test_tensor),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=25, restore_best_weights=True, mode='max', verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1
    ),
    tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
]

print(f"TensorBoard Logs tersimpan di: {log_dir}")

EPOCHS = 200
BATCH_SIZE = 256

# Class weight untuk menangani imbalanced data
# Weight 3.5 pada kelas minoritas (default=1) memberi penalti lebih
# saat model salah memprediksi nasabah yang sebenarnya gagal bayar
history = model.fit(
    X_train_tensor, y_train_tensor,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(X_test_tensor, y_test_tensor),
    callbacks=callbacks, verbose=1,
    class_weight={0: 1.0, 1: 3.5}
)

TensorBoard Logs tersimpan di: logs\20260527-172316
Epoch 1/200
90/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5209 - loss: 1.1974  >> Val Acc: 0.7080 | MAE: 0.4299 | F1: 0.4942 | Best: 0.7080
94/94 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.6221 - loss: 1.1190 - val_accuracy: 0.7080 - val_loss: 0.7620 - learning_rate: 0.0010
Epoch 2/200
91/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7226 - loss: 1.0338  >> Val Acc: 0.7565 | MAE: 0.4114 | F1: 0.5112 | Best: 0.7565
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7262 - loss: 1.0332 - val_accuracy: 0.7565 - val_loss: 0.7084 - learning_rate: 0.0010
Epoch 3/200
89/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7339 - loss: 1.0183  >> Val Acc: 0.7672 | MAE: 0.3961 | F1: 0.5171 | Best: 0.7672
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7378 - loss: 1.0176 - val_accuracy: 0.7672 - val_loss: 0.6893 - learning_rate: 0.0010
Epoch 4/200
86/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7399 - loss: 1.0049  >> V

## 🔄 STEP 4b: Fine-Tuning dengan GradientTape (Custom Loop)
Melakukan fine-tuning lanjutan dengan *learning rate* sangat kecil menggunakan custom *GradientTape* loop untuk menstabilkan dan meningkatkan akurasi model secara detail.

In [11]:
print("Memulai Fine-tuning dengan GradientTape...")

fine_tune_optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
fine_tune_loss = tf.keras.losses.BinaryCrossentropy()
FINE_TUNE_EPOCHS = 10

train_dataset = tf.data.Dataset.from_tensor_slices((X_train_tensor, y_train_tensor))\
    .shuffle(buffer_size=len(X_train_scaled)).batch(256)

pre_ft_pred = model(X_test_tensor, training=False)
pre_ft_acc = accuracy_score(
    y_test.values, (tf.reshape(pre_ft_pred, [-1]).numpy() > 0.5).astype(int)
)
print(f"Pre-finetune accuracy: {pre_ft_acc:.4f}")

best_ft_acc = pre_ft_acc
best_ft_weights = model.get_weights()

for epoch in range(FINE_TUNE_EPOCHS):
    epoch_loss = []
    for batch_x, batch_y in train_dataset:
        with tf.GradientTape() as tape:
            predictions = model(batch_x, training=True)
            loss = fine_tune_loss(batch_y, predictions)
            if model.losses:
                loss = loss + tf.add_n(model.losses)
        
        gradients = tape.gradient(loss, model.trainable_variables)
        gradients, _ = tf.clip_by_global_norm(gradients, 1.0)
        fine_tune_optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        epoch_loss.append(loss.numpy())

    # Evaluasi
    ft_pred = model(X_test_tensor, training=False)
    ft_pred_binary = (tf.reshape(ft_pred, [-1]).numpy() > 0.5).astype(int)
    ft_acc = accuracy_score(y_test.values, ft_pred_binary)
    ft_mae = mean_absolute_error(y_test.values, tf.reshape(ft_pred, [-1]).numpy())
    
    print(f"FT Epoch {epoch+1}/{FINE_TUNE_EPOCHS} | Loss: {np.mean(epoch_loss):.4f} | Acc: {ft_acc:.4f} | MAE: {ft_mae:.4f}")
    
    if ft_acc > best_ft_acc:
        best_ft_acc = ft_acc
        best_ft_weights = model.get_weights()

model.set_weights(best_ft_weights)
print(f"\n✓ Best fine-tune accuracy restored: {best_ft_acc:.4f}")

Memulai Fine-tuning dengan GradientTape...
Pre-finetune accuracy: 0.7747
FT Epoch 1/10 | Loss: 0.6008 | Acc: 0.7797 | MAE: 0.3646
FT Epoch 2/10 | Loss: 0.5747 | Acc: 0.7830 | MAE: 0.3529
FT Epoch 3/10 | Loss: 0.5527 | Acc: 0.7858 | MAE: 0.3420
FT Epoch 4/10 | Loss: 0.5372 | Acc: 0.7912 | MAE: 0.3306
FT Epoch 5/10 | Loss: 0.5272 | Acc: 0.7985 | MAE: 0.3210
FT Epoch 6/10 | Loss: 0.5205 | Acc: 0.8032 | MAE: 0.3133
FT Epoch 7/10 | Loss: 0.5131 | Acc: 0.8055 | MAE: 0.3061
FT Epoch 8/10 | Loss: 0.5085 | Acc: 0.8072 | MAE: 0.3001
FT Epoch 9/10 | Loss: 0.5055 | Acc: 0.8075 | MAE: 0.2951
FT Epoch 10/10 | Loss: 0.5013 | Acc: 0.8078 | MAE: 0.2906

✓ Best fine-tune accuracy restored: 0.8078


## 📈 STEP 5: Final Evaluation
Melakukan evaluasi akhir menggunakan data uji dan mencetak *Classification Report* lengkap.

In [12]:
test_predictions = model(X_test_tensor, training=False)
pred_proba = tf.reshape(test_predictions, [-1]).numpy()
pred_binary = (pred_proba > 0.5).astype(int)
y_test_np = y_test.values.flatten()

final_accuracy = accuracy_score(y_test_np, pred_binary)
final_mae = mean_absolute_error(y_test_np, pred_proba)
final_f1 = f1_score(y_test_np, pred_binary)

print(f"\n{'='*40}")
print(f"  FINAL RESULTS")
print(f"{'='*40}")
print(f"  Accuracy:  {final_accuracy:.4f}")
print(f"  MAE:       {final_mae:.4f}")
print(f"  F1 Score:  {final_f1:.4f}")
print(f"{'='*40}")

if final_accuracy >= 0.85:
    print("  ✅ Accuracy target ACHIEVED (≥ 85%)")
else:
    print(f"  ⚠ Accuracy: {final_accuracy:.2%}")

print(f"\n--- Classification Report ---")
print(classification_report(y_test_np, pred_binary, target_names=['Tidak Gagal Bayar', 'Gagal Bayar']))


  FINAL RESULTS
  Accuracy:  0.8078
  MAE:       0.2906
  F1 Score:  0.4954
  ⚠ Accuracy: 80.78%

--- Classification Report ---
                   precision    recall  f1-score   support

Tidak Gagal Bayar       0.85      0.92      0.88      4673
      Gagal Bayar       0.59      0.43      0.50      1327

         accuracy                           0.81      6000
        macro avg       0.72      0.67      0.69      6000
     weighted avg       0.79      0.81      0.80      6000



## Save Model & Verifikasi
Menyimpan model ke dalam format `.keras` dan mengujinya kembali untuk memastikan model dapat dimuat dengan sempurna di FastAPI.

In [13]:
model_path = 'credit_default_model.keras'
model.save(model_path, overwrite=True)
print(f"✓ Model berhasil disimpan ke {model_path}")

# Verifikasi Load Model
loaded_model = tf.keras.models.load_model(
    model_path,
    custom_objects={
        'CustomDenseBlock': CustomDenseBlock,
    }
)

# Tes prediksi 5 sampel pertama
verify_pred = loaded_model.predict(X_test_tensor[:5])
print(f"\nVerifikasi Prediksi (5 sampel):")
for i in range(5):
    prob = verify_pred[i][0]
    actual = int(y_test_np[i])
    pred = "Gagal Bayar" if prob > 0.5 else "Tidak Gagal Bayar"
    print(f"  Sampel {i+1}: Prob={prob:.4f} | Aktual={actual} | Prediksi={pred}")

print(f"\n{'='*60}")
print("TRAINING COMPLETE")
print(f"{'='*60}")
print(f"Model: {model_path}")
print(f"Scaler: scaler.joblib")
print(f"Features: feature_names.joblib ({len(all_features)} features)")
print(f"{'='*60}")

✓ Model berhasil disimpan ke credit_default_model.keras

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step

Verifikasi Prediksi (5 sampel):
  Sampel 1: Prob=0.1507 | Aktual=0 | Prediksi=Tidak Gagal Bayar
  Sampel 2: Prob=0.2155 | Aktual=0 | Prediksi=Tidak Gagal Bayar
  Sampel 3: Prob=0.1815 | Aktual=0 | Prediksi=Tidak Gagal Bayar
  Sampel 4: Prob=0.1693 | Aktual=1 | Prediksi=Tidak Gagal Bayar
  Sampel 5: Prob=0.1169 | Aktual=0 | Prediksi=Tidak Gagal Bayar

TRAINING COMPLETE
Model: credit_default_model.keras
Scaler: scaler.joblib
Features: feature_names.joblib (31 features)
